# Bài 2: K-Means Clustering - Giải Thích Chi Tiết & Ứng Dụng

**Session 7 - Advanced Data Science with Python**

---

## Mục tiêu bài học

- Hiểu **nguyên lý hoạt động** chi tiết của K-Means
- Nắm vững **cách chọn K** tối ưu (Elbow, Silhouette)
- So sánh K-Means với **DBSCAN, Hierarchical, GMM**
- Biết **khi nào K-Means thất bại** và dùng thuật toán gì thay thế
- Thực hành phân nhóm khách hàng với **dữ liệu thực**

---

## 1. K-Means Hoạt Động Như Thế Nào?

### 1.1 Ý tưởng cốt lõi

K-Means phân chia N điểm dữ liệu vào **K cụm** sao cho:
- Mỗi điểm thuộc **đúng 1 cụm** (hard clustering)
- Mỗi cụm có 1 **tâm (centroid)** = trung bình tất cả điểm trong cụm
- Mỗi điểm được gán vào cụm có **tâm gần nhất** (Euclidean distance)

### 1.2 Hàm mục tiêu

K-Means tối thiểu hóa **Inertia** (Within-Cluster Sum of Squares - WCSS):

$$J = \sum_{k=1}^{K} \sum_{x_i \in C_k} ||x_i - \mu_k||^2$$

- $C_k$: tập điểm thuộc cụm k
- $\mu_k$: tâm cụm k
- Inertia **càng nhỏ** → các điểm trong cụm **càng gần nhau** → càng tốt

### 1.3 Thuật toán từng bước (Lloyd's Algorithm)

```
Input: Dữ liệu X, số cụm K

1. KHỞI TẠO: Chọn K centroids ban đầu (random hoặc K-Means++)

2. LẶP LẠI cho đến khi hội tụ:
   
   a. GÁN CỤM:
      Với mỗi điểm x_i → tính khoảng cách đến K centroids
                        → gán vào cụm có centroid GẦN NHẤT
   
   b. CẬP NHẬT TÂM:
      Với mỗi cụm k → centroid mới = TRUNG BÌNH tất cả điểm trong cụm
   
   c. Nếu centroids KHÔNG thay đổi → DỪNG

Output: K centroids + nhãn cụm cho mỗi điểm
```

### 1.4 K-Means++ là gì?

Vấn đề: Khởi tạo random có thể chọn centroids gần nhau → kết quả tệ.

**K-Means++** giải quyết bằng cách:
1. Chọn centroid đầu tiên ngẫu nhiên
2. Centroid tiếp theo: chọn điểm **CÀng XA** centroids đã chọn (xác suất tỷ lệ với khoảng cách²)
3. Lặp lại cho đến đủ K centroids

→ Centroids ban đầu **trải đều** → hội tụ nhanh hơn, kết quả tốt hơn.

**Trong sklearn**: `init='k-means++'` là **MẶC ĐỊNH** → không cần thay đổi gì.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.cluster import KMeans, DBSCAN, AgglomerativeClustering
from sklearn.mixture import GaussianMixture
from sklearn.datasets import make_blobs, make_moons, make_circles
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import silhouette_score, silhouette_samples, adjusted_rand_score
from sklearn.decomposition import PCA
from scipy.cluster.hierarchy import dendrogram, linkage
import warnings
warnings.filterwarnings('ignore')
plt.rcParams['figure.dpi'] = 100
print("✅ Thư viện đã sẵn sàng!")

In [ ]:
# Minh họa quá trình K-Means qua từng bước
np.random.seed(42)
X_demo, _ = make_blobs(300, centers=3, cluster_std=1.2, random_state=42)

fig, axes = plt.subplots(2, 3, figsize=(18, 10))
colors = ['#e74c3c', '#3498db', '#2ecc71']

# Bước 0: Dữ liệu ban đầu
axes[0,0].scatter(X_demo[:,0], X_demo[:,1], c='gray', s=20, alpha=0.5)
axes[0,0].set_title('Bước 0: Dữ liệu ban đầu\n(chưa có nhãn)', fontweight='bold')

# Chạy K-Means từng bước bằng max_iter
for step in range(1, 6):
    row, col = divmod(step, 3)
    km = KMeans(n_clusters=3, init='random', n_init=1, max_iter=step, random_state=10)
    labels = km.fit_predict(X_demo)
    
    for k in range(3):
        mask = labels == k
        axes[row, col].scatter(X_demo[mask, 0], X_demo[mask, 1], c=colors[k], s=20, alpha=0.5)
    axes[row, col].scatter(km.cluster_centers_[:,0], km.cluster_centers_[:,1], 
                           c='black', s=200, marker='X', edgecolors='white', linewidths=2, zorder=5)
    axes[row, col].set_title(f'Bước {step}: Gán + Cập nhật', fontweight='bold')

plt.suptitle('Quá trình K-Means hội tụ dần\n(X đen = centroids, di chuyển mỗi bước)', 
             fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()
print("→ Centroids di chuyển dần đến tâm thực sự của mỗi cụm cho đến khi ổn định")

---

## 2. Cách Chọn K Tối Ưu

**Đây là câu hỏi quan trọng nhất khi dùng K-Means!**

### 2.1 Elbow Method
- Vẽ Inertia vs K → tìm điểm **"khuỷu tay"** = Inertia bắt đầu giảm chậm
- Trực giác: Thêm cụm nữa **không cải thiện đáng kể**

### 2.2 Silhouette Score (QUAN TRỌNG HƠN)

$$s(i) = \frac{b(i) - a(i)}{\max(a(i), b(i))} \in [-1, 1]$$

- $a(i)$: khoảng cách TB đến các điểm **cùng cụm** (intra-cluster)
- $b(i)$: khoảng cách TB đến các điểm **cụm gần nhất khác** (inter-cluster)

| Giá trị | Ý nghĩa |
|:---|:---|
| **0.7 - 1.0** | Cấu trúc cụm rất rõ ràng |
| **0.5 - 0.7** | Cấu trúc cụm hợp lý |
| **0.25 - 0.5** | Cấu trúc yếu, có thể trùng lặp |
| **< 0.25** | Không có cấu trúc cụm rõ |

In [ ]:
# Demo: Tìm K tối ưu
X_test, _ = make_blobs(500, centers=4, cluster_std=1.0, random_state=42)

K_range = range(2, 10)
inertias, silhouettes = [], []

for k in K_range:
    km = KMeans(k, random_state=42, n_init=10)
    labels = km.fit_predict(X_test)
    inertias.append(km.inertia_)
    silhouettes.append(silhouette_score(X_test, labels))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(list(K_range), inertias, 'bo-', lw=2, ms=8)
axes[0].axvline(x=4, color='red', ls='--', alpha=0.7, label='K=4 (khuỷu tay)')
axes[0].set_xlabel('K'); axes[0].set_ylabel('Inertia')
axes[0].set_title('Elbow Method', fontsize=13, fontweight='bold')
axes[0].legend(); axes[0].grid(alpha=0.3)

axes[1].plot(list(K_range), silhouettes, 'rs-', lw=2, ms=8)
best_k = list(K_range)[np.argmax(silhouettes)]
axes[1].axvline(x=best_k, color='red', ls='--', alpha=0.7, label=f'K={best_k} (max)')
axes[1].set_xlabel('K'); axes[1].set_ylabel('Silhouette')
axes[1].set_title('Silhouette Score (CAO = TỐT)', fontsize=13, fontweight='bold')
axes[1].legend(); axes[1].grid(alpha=0.3)

plt.tight_layout(); plt.show()
print(f"📊 Cả 2 phương pháp đều chỉ ra K = {best_k} (đúng với data có 4 cụm thật)")

In [ ]:
# Silhouette Plot chi tiết - giúp thấy chất lượng TỪNG cụm
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for idx, k in enumerate([3, 4, 5]):
    ax = axes[idx]
    km = KMeans(k, random_state=42, n_init=10)
    labels = km.fit_predict(X_test)
    sil_vals = silhouette_samples(X_test, labels)
    avg = silhouette_score(X_test, labels)
    
    y_lower = 10
    for c in range(k):
        c_sil = np.sort(sil_vals[labels == c])
        y_upper = y_lower + len(c_sil)
        ax.fill_betweenx(np.arange(y_lower, y_upper), 0, c_sil, alpha=0.7)
        ax.text(-0.05, y_lower + len(c_sil)*0.5, str(c), fontsize=10)
        y_lower = y_upper + 10
    
    ax.axvline(x=avg, color='red', ls='--', lw=2, label=f'Avg: {avg:.3f}')
    ax.set_xlabel('Silhouette'); ax.set_title(f'K = {k}', fontsize=13, fontweight='bold')
    ax.legend(fontsize=11)

plt.suptitle('Silhouette Plot - Chất lượng từng cụm', fontweight='bold', fontsize=14)
plt.tight_layout(); plt.show()

print("📌 K=4: Tất cả cụm đều vượt trung bình, kích thước đồng đều → TỐT NHẤT")
print("📌 K=3 hoặc K=5: Có cụm dưới trung bình hoặc kích thước chênh lệch")

---

## 3. Khi Nào K-Means Thất Bại? Dùng Gì Thay Thế?

### K-Means có 4 hạn chế chính:

| Hạn chế | Giải thích | Giải pháp |
|:---|:---|:---|
| Chỉ tìm cụm **hình cầu** | Không xử lý được cụm hình dạng phức tạp | **DBSCAN** |
| Phải chọn **K trước** | Không tự biết có bao nhiêu cụm | **DBSCAN** (tự tìm) |
| **Nhạy outlier** | 1 outlier có thể kéo centroid lệch | **DBSCAN** hoặc xử lý outlier trước |
| **Hard clustering** | Mỗi điểm chỉ thuộc 1 cụm | **GMM** (xác suất) |

In [ ]:
# Demo: K-Means thất bại với dữ liệu phi tuyến
X_moons, y_moons = make_moons(300, noise=0.08, random_state=42)
X_circles, y_circles = make_circles(300, noise=0.05, factor=0.5, random_state=42)

fig, axes = plt.subplots(2, 3, figsize=(18, 10))

for row, (X_data, y_true, name) in enumerate([
    (X_moons, y_moons, 'Moons (trăng lưỡi liềm)'),
    (X_circles, y_circles, 'Circles (vòng tròn lồng)')
]):
    # Ground truth
    axes[row,0].scatter(X_data[:,0], X_data[:,1], c=y_true, cmap='Set1', s=30)
    axes[row,0].set_title(f'Nhãn thật: {name}', fontweight='bold')
    
    # K-Means
    labels_km = KMeans(2, random_state=42, n_init=10).fit_predict(X_data)
    axes[row,1].scatter(X_data[:,0], X_data[:,1], c=labels_km, cmap='Set1', s=30)
    axes[row,1].set_title('❌ K-Means\n(Thất bại!)', color='red', fontweight='bold')
    
    # DBSCAN
    labels_db = DBSCAN(eps=0.2, min_samples=5).fit_predict(X_data)
    axes[row,2].scatter(X_data[:,0], X_data[:,1], c=labels_db, cmap='Set1', s=30)
    axes[row,2].set_title('✅ DBSCAN\n(Thành công!)', color='green', fontweight='bold')

plt.suptitle('K-Means chỉ tìm cụm hình CẦU → Thất bại với dạng phi tuyến', 
             fontsize=14, fontweight='bold')
plt.tight_layout(); plt.show()

---

## 4. So Sánh Các Thuật Toán Clustering

### 4.1 Bảng so sánh tổng quan

| Thuật toán | Ưu điểm | Nhược điểm | Khi nào dùng |
|:---|:---|:---|:---|
| **K-Means** | Nhanh, đơn giản | Chỉ cụm tròn, cần chọn K | Dữ liệu lớn, cụm rõ |
| **DBSCAN** | Tìm cụm bất kỳ, tự tìm K, loại noise | Nhạy eps/min_samples | Cụm hình dạng lạ |
| **Hierarchical** | Dendrogram trực quan | Chậm O(n³) | Dữ liệu nhỏ, cần phân cấp |
| **GMM** | Soft clustering, linh hoạt | Phức tạp hơn K-Means | Cụm overlap |

In [ ]:
# So sánh 4 thuật toán trên cùng dữ liệu
X_comp, y_comp = make_blobs(400, centers=4, cluster_std=[1.0, 1.5, 0.5, 1.2], random_state=42)

fig, axes = plt.subplots(1, 4, figsize=(20, 4.5))

# K-Means
l1 = KMeans(4, random_state=42, n_init=10).fit_predict(X_comp)
axes[0].scatter(X_comp[:,0], X_comp[:,1], c=l1, cmap='Set1', s=20)
axes[0].set_title(f'K-Means\nSil={silhouette_score(X_comp,l1):.3f}', fontweight='bold')

# DBSCAN
l2 = DBSCAN(eps=1.2, min_samples=5).fit_predict(X_comp)
axes[1].scatter(X_comp[:,0], X_comp[:,1], c=l2, cmap='Set1', s=20)
n_clusters_db = len(set(l2)) - (1 if -1 in l2 else 0)
axes[1].set_title(f'DBSCAN (tự tìm {n_clusters_db} cụm)\n+ noise (đen)', fontweight='bold')

# Hierarchical
l3 = AgglomerativeClustering(4).fit_predict(X_comp)
axes[2].scatter(X_comp[:,0], X_comp[:,1], c=l3, cmap='Set1', s=20)
axes[2].set_title(f'Hierarchical\nSil={silhouette_score(X_comp,l3):.3f}', fontweight='bold')

# GMM
l4 = GaussianMixture(4, random_state=42).fit_predict(X_comp)
axes[3].scatter(X_comp[:,0], X_comp[:,1], c=l4, cmap='Set1', s=20)
axes[3].set_title(f'GMM\nSil={silhouette_score(X_comp,l4):.3f}', fontweight='bold')

plt.suptitle('So sánh 4 thuật toán Clustering', fontsize=14, fontweight='bold')
plt.tight_layout(); plt.show()

In [ ]:
# === DBSCAN chi tiết ===
# DBSCAN không cần chọn K, nhưng cần 2 tham số:
# - eps: bán kính neighborhood
# - min_samples: số điểm tối thiểu trong neighborhood để tạo cụm

X_db, _ = make_blobs(300, centers=3, cluster_std=0.8, random_state=42)
# Thêm noise
noise = np.random.uniform(-10, 10, (30, 2))
X_db = np.vstack([X_db, noise])

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for ax, eps in zip(axes, [0.5, 1.0, 2.0]):
    labels = DBSCAN(eps=eps, min_samples=5).fit_predict(X_db)
    n_clusters = len(set(labels)) - (1 if -1 in labels else 0)
    n_noise = (labels == -1).sum()
    
    # Noise points in black
    ax.scatter(X_db[labels==-1, 0], X_db[labels==-1, 1], c='black', s=20, marker='x', label=f'Noise ({n_noise})')
    ax.scatter(X_db[labels>=0, 0], X_db[labels>=0, 1], c=labels[labels>=0], cmap='Set1', s=20)
    ax.set_title(f'eps={eps}\n{n_clusters} cụm, {n_noise} noise', fontweight='bold')
    ax.legend()

plt.suptitle('DBSCAN: Ảnh hưởng của tham số eps', fontsize=14, fontweight='bold')
plt.tight_layout(); plt.show()

print("📌 eps nhỏ → nhiều noise, ít cụm")
print("📌 eps lớn → ít noise, cụm gộp lại")
print("📌 DBSCAN tự động LOẠI BỎ outlier (label = -1)")

In [ ]:
# === Hierarchical Clustering + Dendrogram ===
from scipy.cluster.hierarchy import dendrogram, linkage

# Tạo dữ liệu nhỏ để dendrogram rõ
np.random.seed(42)
X_hier = np.vstack([
    np.random.randn(10, 2) + [0, 0],
    np.random.randn(10, 2) + [5, 5],
    np.random.randn(10, 2) + [10, 0]
])

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Dendrogram
Z = linkage(X_hier, method='ward')  # Ward = minimize variance
dendrogram(Z, ax=axes[0], truncate_mode='lastp', p=12)
axes[0].set_title('Dendrogram (Ward linkage)\nCắt ngang = chọn số cụm', fontweight='bold')
axes[0].axhline(y=15, color='red', ls='--', label='Cắt → 3 cụm')
axes[0].legend()

# Kết quả clustering
labels_h = AgglomerativeClustering(3).fit_predict(X_hier)
axes[1].scatter(X_hier[:,0], X_hier[:,1], c=labels_h, cmap='Set1', s=50)
axes[1].set_title('Hierarchical Clustering (K=3)', fontweight='bold')

plt.tight_layout(); plt.show()
print("📌 Dendrogram cho thấy cấu trúc phân cấp của dữ liệu")
print("📌 Cắt ngang ở vị trí khác nhau → số cụm khác nhau")

In [ ]:
# === GMM: Soft Clustering ===
# GMM cho XÁC SUẤT thuộc mỗi cụm (không chỉ 0/1 như K-Means)

X_gmm, _ = make_blobs(300, centers=3, cluster_std=1.5, random_state=42)
gmm = GaussianMixture(n_components=3, random_state=42)
gmm.fit(X_gmm)

# Xác suất thuộc mỗi cụm
probs = gmm.predict_proba(X_gmm)
labels_gmm = gmm.predict(X_gmm)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Hard labels
axes[0].scatter(X_gmm[:,0], X_gmm[:,1], c=labels_gmm, cmap='Set1', s=30)
axes[0].set_title('GMM: Hard Labels\n(giống K-Means)', fontweight='bold')

# Uncertainty (điểm gần biên có xác suất gần 50/50)
uncertainty = 1 - probs.max(axis=1)
sc = axes[1].scatter(X_gmm[:,0], X_gmm[:,1], c=uncertainty, cmap='RdYlGn_r', s=30)
plt.colorbar(sc, ax=axes[1], label='Uncertainty')
axes[1].set_title('GMM: Uncertainty\n(đỏ = khó xác định thuộc cụm nào)', fontweight='bold')

plt.tight_layout(); plt.show()

print("📌 Ví dụ xác suất điểm 0:", probs[0].round(3))
print("→ GMM biết điểm NÀO 'không chắc chắn' thuộc cụm nào")
print("→ Rất hữu ích khi cụm overlap hoặc cần đánh giá risk")

---

## 5. Bài Thực Hành: Phân Nhóm Khách Hàng Wholesale

**Bối cảnh**: Công ty bán buôn muốn phân nhóm khách hàng để tối ưu marketing.

**Yêu cầu**:
1. EDA + Chuẩn hóa
2. Tìm K tối ưu
3. K-Means clustering
4. Phân tích đặc điểm từng nhóm
5. Đề xuất chiến lược kinh doanh

In [ ]:
# BÀI GIẢI MẪU

# 1. Load & EDA
df = pd.read_csv('../Bai thi thu/Course Files/Wholesale customers data.csv')
features = ['Fresh', 'Milk', 'Grocery', 'Frozen', 'Detergents_Paper', 'Delicassen']

print(f"Shape: {df.shape}")
print(f"\nCorrelation matrix:")
plt.figure(figsize=(8, 6))
sns.heatmap(df[features].corr(), annot=True, cmap='RdBu_r', center=0, fmt='.2f')
plt.title('Feature Correlations', fontweight='bold')
plt.tight_layout(); plt.show()

print("→ Milk, Grocery, Detergents_Paper tương quan mạnh")
print("→ Gợi ý: có 1 nhóm khách hàng mua nhiều 3 thứ này cùng lúc")

In [ ]:
# 2. Tiền xử lý: Log transform + StandardScaler
# Log vì dữ liệu chi tiêu thường lệch phải (right-skewed)
X_log = np.log1p(df[features])  # log(1+x) tránh log(0)
X_scaled = StandardScaler().fit_transform(X_log)

# 3. Tìm K
results = []
for k in range(2, 8):
    km = KMeans(k, random_state=42, n_init=10)
    l = km.fit_predict(X_scaled)
    results.append({'K': k, 'Silhouette': silhouette_score(X_scaled, l), 'Inertia': km.inertia_})

res = pd.DataFrame(results)
fig, ax = plt.subplots(figsize=(8, 4))
ax.bar(res['K'], res['Silhouette'], color='steelblue', alpha=0.8)
ax.set_xlabel('K'); ax.set_ylabel('Silhouette'); ax.set_title('Silhouette Score', fontweight='bold')
for _, r in res.iterrows():
    ax.text(r['K'], r['Silhouette']+0.005, f"{r['Silhouette']:.3f}", ha='center')
plt.tight_layout(); plt.show()

best_k = int(res.loc[res['Silhouette'].idxmax(), 'K'])
print(f"🏆 K tối ưu: {best_k}")

In [ ]:
# 4. Clustering & Phân tích
km_final = KMeans(best_k, random_state=42, n_init=10)
df['Cluster'] = km_final.fit_predict(X_scaled)

# Profile từng nhóm
profile = df.groupby('Cluster')[features].mean().round(0)
print("Đặc điểm trung bình:")
print(profile.to_string())
print(f"\nSố lượng: {df['Cluster'].value_counts().sort_index().to_dict()}")

# Radar chart
from math import pi
cluster_norm = (profile - profile.min()) / (profile.max() - profile.min())
angles = [n / len(features) * 2 * pi for n in range(len(features))] + [0]

fig, ax = plt.subplots(figsize=(8, 8), subplot_kw=dict(polar=True))
colors = ['#e74c3c', '#3498db', '#2ecc71', '#f39c12']
for c in cluster_norm.index:
    vals = cluster_norm.loc[c].tolist() + [cluster_norm.loc[c].tolist()[0]]
    ax.plot(angles, vals, 'o-', lw=2, label=f'Nhóm {c}', color=colors[c])
    ax.fill(angles, vals, alpha=0.1, color=colors[c])
ax.set_xticks(angles[:-1]); ax.set_xticklabels(features, fontsize=9)
ax.set_title('Radar Chart: Đặc điểm từng nhóm', fontweight='bold', pad=20)
ax.legend(loc='upper right', bbox_to_anchor=(1.3, 1.1))
plt.tight_layout(); plt.show()

In [ ]:
# PCA visualization
pca = PCA(2)
X_pca = pca.fit_transform(X_scaled)

plt.figure(figsize=(10, 7))
for c in sorted(df['Cluster'].unique()):
    m = df['Cluster'] == c
    plt.scatter(X_pca[m,0], X_pca[m,1], label=f'Nhóm {c} (n={m.sum()})', s=40, alpha=0.7)
plt.xlabel(f'PC1 ({pca.explained_variance_ratio_[0]*100:.1f}%)')
plt.ylabel(f'PC2 ({pca.explained_variance_ratio_[1]*100:.1f}%)')
plt.title('Phân nhóm khách hàng (Wholesale)', fontsize=14, fontweight='bold')
plt.legend(fontsize=11); plt.grid(alpha=0.3); plt.show()

# 5. Đề xuất
print("\n" + "="*60)
print("📊 ĐỀ XUẤT CHIẾN LƯỢC:")
print("="*60)
for c in sorted(df['Cluster'].unique()):
    top = profile.loc[c].nlargest(2).index.tolist()
    print(f"  Nhóm {c}: Cao nhất {', '.join(top)} → Tập trung marketing {', '.join(top)}")

---

## 📌 TỔNG HỢP: 20% Kiến Thức → 80% Ứng Dụng

| # | Kiến thức cốt lõi | Chi tiết |
|:--|:---|:---|
| 1 | **K-Means = Gán cụm + Cập nhật centroid** | 2 bước lặp đến hội tụ. `init='k-means++'` mặc định |
| 2 | **Chọn K bằng Silhouette Score** | > 0.5 tốt. Kết hợp Elbow + domain knowledge |
| 3 | **K-Means chỉ cho cụm tròn** | Phi tuyến → DBSCAN. Overlap → GMM |
| 4 | **Luôn `StandardScaler()`** trước clustering | Và `np.log1p()` nếu dữ liệu skewed |
| 5 | **Flowchart chọn thuật toán** | Cụm rõ+lớn→K-Means, Phi tuyến→DBSCAN, Phân cấp→Hierarchical, Overlap→GMM |

### Code Template:
```python
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import silhouette_score

X_scaled = StandardScaler().fit_transform(X)

# Tìm K
for k in range(2, 10):
    labels = KMeans(k, n_init=10, random_state=42).fit_predict(X_scaled)
    print(f'K={k}: Sil={silhouette_score(X_scaled, labels):.3f}')

# Fit
final = KMeans(best_k, n_init=10, random_state=42).fit_predict(X_scaled)
```

---
**→ Bài tiếp theo: [Bài 3] PCA - Principal Component Analysis**